In [3]:
import os
import pandas as pd
import numpy as np
import yfinance as yf

In [5]:
path_transactions = os.path.join(os.path.dirname(os.getcwd()),'data/Transactions.csv')
path_account = os.path.join(os.path.dirname(os.getcwd()),'data/Account.csv')

isin_to_ticker = {
    'IE00B4K48X80': 'IMAE.AS',
    'NL0010273215': 'ASML.AS',
    'IE00B4L5Y983': 'IWDA.L',
    'NL0013654783': 'PRX.AS',
    'DE0007664039': 'VOW3.DE',
    'NL0012866412': 'BESI.AS',
    'US6541061031': 'NKE',
    'NL0012015705': 'TKWY.AS',
    'BMG3602E1084': 'FLOW.AS',
    'US01609W1027': 'BABA',
    'GB00BP6MXD84': 'SHEL.L',
    'NL0011279492': 'FLOW.AS',
    'US64110L1061': 'NFLX',
    'NL0011794037': 'AD.AS',
    'GB00B03MLX29': 'RDSA',
    'NL0000009538': 'PHIA.AS',
    'US00206R1023': 'T',
    'GB00B10RZP78': 'ULVR.L',
    'NL0011872650': 'BFIT.AS',
    'IE00BJGWQN72': 'WCLD.L',
    'NL0010391025': 'PHARM.AS',
    'NL0011540547': 'ABN.AS',
    'US7170811035': 'PFE',
    'BE0003818359': 'GLPG.AS',
    'US92556V1061': 'VTRS',
    'NL0000334118': 'ASM.AS',
    'NL0000440584': 'ORDI',
    'NL0011821202': 'INGA.AS'
}

ticker_to_currency = {
    'IMAE.AS': '',
    'ASML.AS': '',
    'IWDA.L': '',
    'PRX.AS': '',
    'VOW3.DE': '',
    'BESI.AS': '',
    'NKE': '',
    'TKWY.AS': '',
    'FLOW.AS': '', # Keeping both entries for FLOW.AS
    'FLOW.AS': '',
    'BABA': '',
    'SHEL.L': '',
    'NFLX': '',
    'AD.AS': '',
    'RDSA': '',
    'PHIA.AS': '',
    'T': '',
    'ULVR.L': '',
    'BFIT.AS': '',
    'WCLD.L': '',
    'PHARM.AS': '',
    'ABN.AS': '',
    'PFE': '',
    'GLPG.AS': '',
    'VTRS': '',
    'ASM.AS': '',
    'ORDI': '',
    'INGA.AS': ''
}

In [22]:
ideal = ['iDEAL Deposit',
'iDEAL storting']


df = pd.read_csv(path_account)

# data cleaning
df.columns = ['Datum', 'Tijd', 'Valutadatum', 'Product', 'ISIN', 'Omschrijving', 'FX',
       'Valuta Mutatie', 'Waarde Mutatie', 'Valuta Saldo', 'Waarde Saldo', 'Order Id']

df = df.drop(['Valuta Saldo', 'Waarde Saldo', 'Order Id'],axis=1)


df_storting = df.loc[df['Omschrijving'].isin(ideal)]
                     
df_storting['Waarde Mutatie'] = df_storting['Waarde Mutatie'].str.replace(',','.').astype(float)

df_storting['Waarde Mutatie'].sum()




/var/folders/c4/p_rq7mxn2knb64hrrwwdz4dc0000gn/T/ipykernel_946/121255365.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_storting['Waarde Mutatie'] = df_storting['Waarde Mutatie'].str.replace(',','.').astype(float)


np.float64(45966.119999999995)

In [19]:
def clean_data(datapath:str) -> pd.DataFrame:
    """cleans stock data to the minimum"""
    df = pd.read_csv(datapath)
    df['Datum'] = pd.to_datetime(df['Datum'], format='%d-%m-%Y')
    df['Year'] = df['Datum'].dt.year
    df = df[['Datum','Year','Tijd','Product','ISIN','Aantal','Totaal']]
    df = df.rename({'Datum':'Date',
                    'Tijd': 'Time',
                    'Aantal':'N',
                    'Totaal':'Total'},axis=1)
    df['trade'] = np.where(df['N']<0,'SELL','BUY')
    df['Total'] = np.abs(df['Total'])
    df['N'] = np.abs(df['N'])
    df['Price'] = df['Total']/ df['N']
    df['Ticker'] = df['ISIN'].replace(isin_to_ticker)
    return df.sort_values(by='Date',ascending=True).reset_index(drop=True)




In [21]:
df = clean_data(datapath)

df

,Date,Year,Time,Product,ISIN,N,Total,trade,Price,Ticker
0,2020-09-21,2020,09:00,ROYAL DUTCH SHELLA,GB00B03MLX29,25,286.04,BUY,11.441600,RDSA
1,2020-09-21,2020,12:40,ING GROEP N.V.,NL0011821202,11,67.69,BUY,6.153636,INGA.AS
2,2020-09-21,2020,12:40,ING GROEP N.V.,NL0011821202,29,173.18,BUY,5.971724,INGA.AS
3,2020-09-28,2020,09:04,GALAPAGOS,BE0003818359,3,385.21,BUY,128.403333,GLPG.AS
4,2020-09-28,2020,09:10,GALAPAGOS,BE0003818359,1,126.29,BUY,126.290000,GLPG.AS
...,...,...,...,...,...,...,...,...,...,...
146,2024-09-04,2024,09:09,ISHARES CORE MSCI WORLD UCITS ETF USD (ACC),IE00B4L5Y983,54,5049.73,BUY,93.513519,IWDA.L
147,2024-10-15,2024,17:16,ASML HOLDING,NL0010273215,1,694.70,BUY,694.700000,ASML.AS
148,2024-10-21,2024,16:08,ISHARES MSCI EUR A,IE00B4K48X80,18,1455.85,BUY,80.880556,IMAE.AS
149,2024-11-01,2024,10:37,ISHARES MSCI EUR A,IE00B4K48X80,13,1020.98,BUY,78.536923,IMAE.AS


In [24]:
year = 2021

df_2020 = df.loc[df['Year']<=year]

df_2020


,Date,Year,Time,Product,ISIN,N,Total,trade,Price,Ticker
0,2020-09-21,2020,09:00,ROYAL DUTCH SHELLA,GB00B03MLX29,25,286.04,BUY,11.441600,RDSA
1,2020-09-21,2020,12:40,ING GROEP N.V.,NL0011821202,11,67.69,BUY,6.153636,INGA.AS
2,2020-09-21,2020,12:40,ING GROEP N.V.,NL0011821202,29,173.18,BUY,5.971724,INGA.AS
3,2020-09-28,2020,09:04,GALAPAGOS,BE0003818359,3,385.21,BUY,128.403333,GLPG.AS
4,2020-09-28,2020,09:10,GALAPAGOS,BE0003818359,1,126.29,BUY,126.290000,GLPG.AS
...,...,...,...,...,...,...,...,...,...,...
87,2021-10-28,2021,12:01,FLOWTRADERS,NL0011279492,35,1092.23,BUY,31.206571,FLOW.AS
88,2021-10-28,2021,11:55,PHILIPS KON,NL0000009538,27,1090.90,SELL,40.403704,PHIA.AS
89,2021-12-06,2021,09:05,TAKEAWAY,NL0012015705,8,401.08,BUY,50.135000,TKWY.AS
90,2021-12-17,2021,09:36,PROSUS,NL0013654783,15,1025.31,BUY,68.354000,PRX.AS


In [ ]:
df_shell

0      2020
1      2020
2      2020
3      2020
4      2020
       ... 
146    2024
147    2024
148    2024
149    2024
150    2025
Name: Date, Length: 151, dtype: int32

In [8]:
def get_stock_price(stock, date):
    stock_data = yf.download(stock, start=date,period='2d')
    return stock_data

get_stock_price('SHEL.L',date ='2020-09-21')

[*********************100%***********************]  1 of 1 completed

1 Failed download:
['SHEL.L']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


Price,Adj Close,Close,High,Low,Open,Volume
Ticker,SHEL.L,SHEL.L,SHEL.L,SHEL.L,SHEL.L,SHEL.L
Date,,,,,,


In [64]:
df['Product'].unique()

array(['ISHARES MSCI EUR A', 'ASML HOLDING',
       'ISHARES CORE MSCI WORLD UCITS ETF USD (ACC)', 'PROSUS',
       'VOLKSWAGEN AG PREFERRED', 'BE SEMICONDUCTOR',
       'NIKE INC. COMMON STOC', 'TAKEAWAY', 'FLOW TRADERS LTD',
       'ADR ON ALIBABA GROUP HOLDING', 'SHELL PLC', 'FLOWTRADERS',
       'NETFLIX INC. - COMMON', 'AHOLD DELHAIZE', 'ROYAL DUTCH SHELLA',
       'PHILIPS KON', 'AT&T INC.', 'UNILEVER', 'BASIC-FIT N.V.',
       'WISDOMTREE CLOUD COMPUTING UCITS ETF USD ACC', 'PHARMING GROUP',
       'ABN AMRO GROUP', 'PFIZER INC. COMMON ST', 'GALAPAGOS',
       'VIATRIS INC. - COMMON STOCK', 'ASM INTERNATIONAL', 'ORDINA NV',
       'ING GROEP N.V.'], dtype=object)